# LLM-assisted metaphor identification: Fine-tuning

This notebook reproduces the fine-tuning experiments reported in the following paper: Fuoli, M., Huang, W., Littlemore, J., Turner, S., & Wilding, E. (2026). Metaphor identification using large language models: A comparison of RAG, prompt engineering, and fine-tuning. *Applied Corpus Linguistics*, DOI: [10.1016/j.acorp.2026.100204](https://doi.org/10.1016/j.acorp.2026.100204).

## Overview

In this study, we evaluate three large language model (LLM) approaches for identifying and labeling metaphorical expressions in a corpus of film reviews:

1. Retrieval-Augmented Generation (RAG)
2. Prompt engineering
3. Fine-tuning

This notebook provides a step-by-step guide to replicating the **fine-tuning** experiments using OpenAI's API. Fine-tuning trains a private copy of a base model (here `gpt-5-mini`) on your labelled examples so that it learns to produce `<Metaphor>` tags without needing a lengthy prompt at inference time.

## What this notebook does

The notebook walks you through the full pipeline, from data preparation to evaluation. Specifically, it:

1. Loads the dataset (`metaphor_dataset_mini.csv`) and splits it into train and test sets
2. Loads the system prompt used for fine-tuning (`prompts.csv`)
3. Converts the training split into JSONL format and uploads it to the OpenAI Files API
4. Submits a fine-tuning job and polls until it completes
5. Saves the fine-tuned model ID so you can reuse it later
6. Applies the fine-tuned model to every text sample in the test set
7. Saves the model outputs to a CSV file
8. Computes evaluation metrics (precision, recall, accuracy, and F1 score) using `evaluation.py`
9. Saves the evaluation results to CSV files

The notebook is written for non-experts. Each section explains what it is doing before the code runs.

## Before you run anything

### Check that all required files are available

Make sure these files are in the same folder as this notebook:

- `metaphor_dataset_mini.csv`
- `prompts.csv`
- `evaluation.py`

### Add your OpenAI API key

The easiest approach is to paste your API key into the next code cell.

### Safer alternative

A safer option is to store your key locally as an environment variable called `OPENAI_API_KEY` and **not** paste it into the notebook.

That method is preferred for real projects, but it is optional here. The example code is included below as commented-out code so it will not run unless you choose to use it.

On macOS or Linux, you can set your API key in a terminal before launching Jupyter:

```bash
export OPENAI_API_KEY="your_api_key_here"
```

On Windows PowerShell:

```powershell
$env:OPENAI_API_KEY="your_api_key_here"
```

Then restart VS Code or Jupyter so the notebook can see the key.

### ⚠️ Important warnings ⚠️

- Your API key is private. Do **not** share it with anyone.
- Do **not** upload or email this notebook with your key pasted into it.
- Using the OpenAI API costs money. Fine-tuning charges both **training tokens** and **inference tokens**.
- Check that your OpenAI account has enough credit or funds before you run the notebook.
- You are responsible for any charges incurred through your API key.
- We cannot take responsibility for money spent through your OpenAI account.

In [ ]:
# -------------------------------
# OpenAI API key setup
# -------------------------------
# Paste your OpenAI API key between the quotes below.
# Example:
# API_KEY = "sk-abc123..."

API_KEY = "PASTE_YOUR_OPENAI_API_KEY_HERE"

# Quick check so the notebook stops early if no key has been added.
if API_KEY == "PASTE_YOUR_OPENAI_API_KEY_HERE":
    raise ValueError(
        "Please paste your OpenAI API key into the API_KEY variable in this cell before running the notebook."
    )
else:
    print("API key added. You are ready to continue.")

# -------------------------------
# Safer alternative (optional)
# -------------------------------
# Instead of pasting your key into the notebook, you can store it locally
# as an environment variable called OPENAI_API_KEY.
#
# Example:
#
# import os
# API_KEY = os.getenv("OPENAI_API_KEY")
#
# if not API_KEY:
#     raise ValueError(
#         "OPENAI_API_KEY was not found in your environment."
#     )
#
# This method is safer because the key stays outside the notebook file.


## 1. Install and import the required packages

The first code cell installs any Python packages that this notebook needs. 
The second cell imports them into Python so the rest of the notebook can use them.


In [ ]:
# pip install -q openai pandas numpy scikit-learn tqdm ipykernel


In [ ]:
import io
import json
import os
import re
import time
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

import sklearn


## 2. Set file paths and experiment settings

This cell defines:

- where the dataset and prompt files live
- which OpenAI **base** model to fine-tune
- what fraction of the data to use for training vs. testing
- how many times to run the fine-tuned model on each test text
- where to save the outputs
- an optional row limit for quick testing

In our study, we ran each model five times on each text and then aggregated the results. While this is not strictly necessary, it improves robustness, as LLM outputs may vary slightly across runs.

For a quick trial run, you can set `ROW_LIMIT` to a small number such as `2` or `3`.

**Note on `TRAIN_FRACTION`:** The fine-tuning API requires a minimum of **10 training examples**. If your dataset is small, either lower `TRAIN_FRACTION` to keep more rows in the training split, or use the full dataset for training and evaluate on the training set itself (set `TRAIN_FRACTION = 1.0`).


In [ ]:
# ---------- File paths ----------
DATASET_PATH    = Path("metaphor_dataset_mini.csv")
PROMPTS_PATH    = Path("prompts.csv")
EVALUATION_PATH = Path("evaluation.py")
OUTPUT_DIR      = Path("outputs")

# ---------- Model ----------
# Base model to fine-tune. Must be a fine-tuning-capable model.
# See https://platform.openai.com/docs/guides/fine-tuning for supported models.
BASE_MODEL_NAME = "gpt-5-mini"

# ---------- Train / test split ----------
# Fraction of the dataset used for training. The rest is used for evaluation.
# Must be between 0.0 (exclusive) and 1.0 (inclusive).
# Set to 1.0 to train on the full dataset and evaluate on the same data.
TRAIN_FRACTION = 0.8
RANDOM_SEED    = 42

# ---------- Fine-tuning hyperparameters ----------
# Leave as None to let OpenAI choose automatically.
# Set to an integer (e.g. 3) to override.
N_EPOCHS = None

# ---------- Optional model name suffix ----------
# A short label appended to the fine-tuned model name in the OpenAI dashboard.
# Must be alphanumeric with hyphens only. Leave as None to skip.
FINE_TUNED_MODEL_SUFFIX = "metaphor"

# ---------- Inference repeats ----------
# Number of times to run the fine-tuned model on each test row.
# Set this to 5 to reproduce the original experiment design.
NUM_REPEATS = 2

# ---------- Optional testing controls ----------
# Set ROW_LIMIT to a small number like 2 or 3 for a quick test.
# Leave as None to run the full dataset.
ROW_LIMIT = None

# ---------- API pacing ----------
# Small pause between inference requests to reduce the chance of rate-limit issues.
SLEEP_BETWEEN_REQUESTS_SECONDS = 0.2

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())
print("Base model:      ", BASE_MODEL_NAME)
print("Train fraction:  ", TRAIN_FRACTION)
print("Repeats per text:", NUM_REPEATS)


## 3. Load the dataset and the system prompt

The dataset contains the texts to annotate.

Expected dataset columns:

- `textid`: a unique identifier for each text
- `plain`: the raw text sent to the model
- `metaphor_tagged_text`: the gold-standard text with `<Metaphor>` tags

The prompt file (`prompts.csv`) contains the instruction messages used to tell the model what task to perform. For fine-tuning we use **only the first prompt strategy** (lowest `pid`) as the system instruction that accompanies every training example. This keeps the fine-tuning signal focused and consistent.

The prompt CSV must contain the same columns as in the prompt engineering notebook:
- `pid`: prompt strategy ID
- `name`: human-readable label
- `cid`: message order within the strategy
- `role`: `system`, `user`, or `assistant`
- `content`: message text


In [ ]:
dataset_df = pd.read_csv(DATASET_PATH)
prompts_df = pd.read_csv(PROMPTS_PATH)

if ROW_LIMIT is not None:
    dataset_df = dataset_df.head(ROW_LIMIT).copy()

required_dataset_cols = {"textid", "plain", "metaphor_tagged_text"}
missing_dataset_cols = required_dataset_cols - set(dataset_df.columns)
if missing_dataset_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_dataset_cols}")

required_prompt_cols = {"pid", "name", "cid", "role", "content"}
missing_prompt_cols = required_prompt_cols - set(prompts_df.columns)
if missing_prompt_cols:
    raise ValueError(f"Prompt file is missing required columns: {missing_prompt_cols}")

print(f"Dataset rows loaded:     {len(dataset_df)}")
print(f"Prompt rows loaded:      {len(prompts_df)}")
print(f"Unique prompt strategies:{prompts_df['pid'].nunique()}")


### Quick preview

These cells let you inspect the data before any processing.


In [ ]:
dataset_df.head()


In [ ]:
prompts_df.head(10)


## 4. Split the dataset into training and test sets

Fine-tuning requires a dedicated **training set** that is uploaded to the OpenAI API. We then evaluate the fine-tuned model on a separate **test set** that the model has never seen during training.

The split is controlled by `TRAIN_FRACTION` and `RANDOM_SEED` defined in section 2. A random seed is used so that the split is reproducible across runs.

**Minimum training set size:** the OpenAI fine-tuning API requires at least 10 training examples. If the training split is too small, increase `TRAIN_FRACTION` or reduce `ROW_LIMIT`.


In [ ]:
train_df = dataset_df.sample(frac=TRAIN_FRACTION, random_state=RANDOM_SEED)
test_df  = dataset_df.drop(train_df.index).reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

MIN_TRAIN_EXAMPLES = 10
if len(train_df) < MIN_TRAIN_EXAMPLES:
    raise ValueError(
        f"Training split has only {len(train_df)} rows. "
        f"The OpenAI fine-tuning API requires at least {MIN_TRAIN_EXAMPLES}. "
        "Increase TRAIN_FRACTION or reduce ROW_LIMIT."
    )

print(f"Training examples: {len(train_df)}")
print(f"Test examples:     {len(test_df)}")


## 5. Extract the system instruction for fine-tuning

Every fine-tuning example is formatted as a mini conversation:

```
{"messages": [
    {"role": "system",    "content": "<task instructions>"},
    {"role": "user",      "content": "<plain text>"},
    {"role": "assistant", "content": "<gold tagged text>"}
]}
```

We extract the **system message** from the first prompt strategy in `prompts.csv`. If the first strategy has no `system` role, we fall back to all `system` rows from that strategy. If there are no system rows at all, a generic fallback instruction is used.

The `user` and `assistant` turns are filled in automatically from the dataset columns `plain` and `metaphor_tagged_text` respectively.


In [ ]:
def extract_system_instruction(prompts_table: pd.DataFrame) -> str:
    first_pid = prompts_table.sort_values("pid")["pid"].iloc[0]
    first_strategy = prompts_table[prompts_table["pid"] == first_pid].sort_values("cid")
    system_rows = first_strategy[first_strategy["role"] == "system"]
    if not system_rows.empty:
        return " ".join(system_rows["content"].tolist()).strip()
    fallback = (
        "You are a linguistic annotator. "
        "Read the text and wrap every metaphorical expression in <Metaphor>...</Metaphor> tags. "
        "Return the full text with tags inserted and nothing else."
    )
    print("Warning: no system row found in prompts.csv. Using fallback instruction.")
    return fallback


SYSTEM_INSTRUCTION = extract_system_instruction(prompts_df)
print("System instruction (first 200 chars):")
print(SYSTEM_INSTRUCTION[:200])


## 6. Create the OpenAI client

This cell creates the client that will talk to the OpenAI API. It is used both for the fine-tuning job (sections 6–8) and for running inference on the fine-tuned model (sections 9–11).

It uses the `API_KEY` value from the cell near the top of the notebook.


In [ ]:
client = OpenAI(api_key=API_KEY)
print("OpenAI client created successfully.")


Or if you are using a local model host (e.g. ollama) / remote non-OpenAI model host that is described as "openai compatible", you may create the client with your customised base URL and headers. Here is an example for the OpenRouter API.


In [ ]:
# client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=API_KEY,
#     default_headers={
#         "HTTP-Referer": "http://localhost:5000",  # Required by OpenRouter
#         "X-Title": "MetaphorIdentification",       # Optional — your app name
#     }
# )

# print("OpenAI client created successfully.")


For a local model e.g. ollama, this is an example:


In [ ]:
# client = OpenAI(
#     base_url="http://localhost:11434/v1",
#     api_key="ollama",  # Required by the SDK, but Ollama ignores it
# )


Please note: when choosing a non-OpenAI model, you need to change `BASE_MODEL_NAME` and verify that the host supports the fine-tuning endpoints used in this notebook.


## 7. Convert the training data to JSONL and upload it

OpenAI's fine-tuning API expects training data as a **JSONL** (JSON Lines) file, where each line is a valid JSON object representing one training example.

Each example follows the chat-completions message format:

```json
{"messages": [
    {"role": "system",    "content": "..."},
    {"role": "user",      "content": "..."},
    {"role": "assistant", "content": "..."}
]}
```

After building the JSONL, we upload it to the OpenAI Files API with `purpose="fine-tune"` and then wait for the file status to reach `"processed"` before submitting the training job. The file is also saved locally to `outputs/fine_tune_train.jsonl` so you can inspect it.


In [ ]:
def build_training_jsonl(df: pd.DataFrame, system_instruction: str) -> str:
    lines = []
    for _, row in df.iterrows():
        example = {
            "messages": [
                {"role": "system",    "content": system_instruction},
                {"role": "user",      "content": row["plain"]},
                {"role": "assistant", "content": row["metaphor_tagged_text"]},
            ]
        }
        lines.append(json.dumps(example, ensure_ascii=False))
    return "\n".join(lines)


train_jsonl_str = build_training_jsonl(train_df, SYSTEM_INSTRUCTION)

# Save locally for inspection.
local_jsonl_path = OUTPUT_DIR / "fine_tune_train.jsonl"
local_jsonl_path.write_text(train_jsonl_str, encoding="utf-8")
print(f"Training JSONL saved to: {local_jsonl_path.resolve()}")
print(f"Lines in JSONL: {train_jsonl_str.count(chr(10)) + 1}")
print("\nFirst example preview:")
print(train_jsonl_str.split("\n")[0][:300])


In [ ]:
# Upload to the OpenAI Files API.
upload_response = client.files.create(
    file=("fine_tune_train.jsonl", io.BytesIO(train_jsonl_str.encode("utf-8")), "application/jsonl"),
    purpose="fine-tune",
)
training_file_id = upload_response.id
print(f"Uploaded training file. File ID: {training_file_id}")

# Wait for the file to be processed before submitting the fine-tuning job.
print("Waiting for file to be processed...", end="", flush=True)
while True:
    file_status = client.files.retrieve(training_file_id).status
    if file_status == "processed":
        print(" done.")
        break
    elif file_status == "error":
        raise RuntimeError(f"File processing failed. File ID: {training_file_id}")
    print(".", end="", flush=True)
    time.sleep(5)

print(f"File status: {file_status}")


## 8. Submit the fine-tuning job

This cell submits a fine-tuning job to the OpenAI API using the uploaded training file. OpenAI will queue the job and begin training asynchronously on its servers.

Key parameters:

- `training_file`: the file ID returned in the previous step
- `model`: the base model to fine-tune (e.g. `gpt-5-mini`)
- `hyperparameters.n_epochs`: number of training passes. Leave as `"auto"` to let OpenAI pick a sensible default based on your dataset size.
- `suffix`: a short label appended to the fine-tuned model name in your dashboard

After this cell runs, the fine-tuning job ID is printed. You can use this ID to check the job status in the next section, even if you close and reopen the notebook.


In [ ]:
hyperparameters = {}
if N_EPOCHS is not None:
    hyperparameters["n_epochs"] = N_EPOCHS

ft_job = client.fine_tuning.jobs.create(
    training_file=training_file_id,
    model=BASE_MODEL_NAME,
    hyperparameters=hyperparameters if hyperparameters else {"n_epochs": "auto"},
    suffix=FINE_TUNED_MODEL_SUFFIX,
)

ft_job_id = ft_job.id
print(f"Fine-tuning job submitted.")
print(f"  Job ID:  {ft_job_id}")
print(f"  Status:  {ft_job.status}")
print(f"  Model:   {ft_job.model}")


## 9. Wait for the fine-tuning job to complete

Fine-tuning typically takes a few minutes to several hours depending on the dataset size and the queue length on OpenAI's servers. This cell polls the job status every 30 seconds until the job either succeeds or fails.

**If you need to interrupt the notebook and resume later**, copy the job ID printed in the previous cell and paste it into `ft_job_id` in the cell below, then re-run from section 9 onwards.

The possible job statuses are:

- `validating_files` — OpenAI is checking your JSONL
- `queued` — waiting for a training slot
- `running` — training in progress
- `succeeded` — training finished successfully
- `failed` — training failed (check the error message)
- `cancelled` — you or OpenAI cancelled the job


In [ ]:
# -----------------------------------------------------------------------
# If you closed the notebook and want to resume from a saved job ID,
# comment out the ft_job_id assignment above and paste your ID here:
# ft_job_id = "ftjob-XXXXXXXXXXXXXXXXXXXXXXXX"
# -----------------------------------------------------------------------

POLL_INTERVAL_SECONDS = 30
TERMINAL_STATUSES = {"succeeded", "failed", "cancelled"}

print(f"Polling fine-tuning job {ft_job_id}...")
while True:
    job_info = client.fine_tuning.jobs.retrieve(ft_job_id)
    status   = job_info.status
    print(f"  [{time.strftime('%H:%M:%S')}] Status: {status}")
    if status in TERMINAL_STATUSES:
        break
    time.sleep(POLL_INTERVAL_SECONDS)

if status == "succeeded":
    FINE_TUNED_MODEL_NAME = job_info.fine_tuned_model
    print(f"\nFine-tuning succeeded!")
    print(f"Fine-tuned model ID: {FINE_TUNED_MODEL_NAME}")
elif status == "failed":
    error_info = getattr(job_info, "error", None)
    raise RuntimeError(
        f"Fine-tuning job failed.\n"
        f"Job ID: {ft_job_id}\n"
        f"Error: {error_info}"
    )
else:
    raise RuntimeError(f"Fine-tuning job ended with unexpected status: {status}")


## 10. Save the fine-tuned model ID

The fine-tuned model ID (e.g. `ft:gpt-5-mini:org:metaphor:XXXXXXXX`) is saved to a small text file so you can retrieve it later without re-running the fine-tuning pipeline.


In [ ]:
model_id_path = OUTPUT_DIR / "fine_tuned_model_id.txt"
model_id_path.write_text(FINE_TUNED_MODEL_NAME, encoding="utf-8")
print(f"Fine-tuned model ID saved to: {model_id_path.resolve()}")
print(f"Model ID: {FINE_TUNED_MODEL_NAME}")


## 11. Helper functions for running inference on the fine-tuned model

These helper functions do most of the heavy lifting for the inference phase.

They:

- send each test text to the fine-tuned model via the OpenAI chat completions API
- collect the model output
- record token usage for each request
- estimate the request cost based on the model used
- repeat each inference as many times as you specify in `NUM_REPEATS`

The pricing table below covers the GPT-5 model family. Fine-tuned model inference is billed at the same rate as the base model. If OpenAI changes its pricing in the future, update the table before re-running cost calculations.


In [ ]:
# Official API prices per 1M tokens for selected text models.
# Update these values if OpenAI changes its pricing in the future.
MODEL_PRICING_PER_1M_TOKENS = {
    # --- Core GPT-5 family ---
    "gpt-5":         {"input": 1.25,  "output": 10.00},
    "gpt-5-mini":    {"input": 0.25,  "output":  2.00},
    "gpt-5-nano":    {"input": 0.05,  "output":  0.40},

    # --- GPT-5.1 series ---
    "gpt-5.1":       {"input": 1.25,  "output": 10.00},
    "gpt-5.1-mini":  {"input": 0.25,  "output":  2.00},

    # --- GPT-5.2 series ---
    "gpt-5.2":       {"input": 1.75,  "output": 14.00},
    "gpt-5.2-mini":  {"input": 0.25,  "output":  2.00},

    # --- GPT-5.3 series ---
    "gpt-5.3":       {"input": 1.75,  "output": 14.00},

    # --- GPT-5.4 (latest flagship) ---
    "gpt-5.4":       {"input": 2.50,  "output": 15.00},
    "gpt-5.4-mini":  {"input": 0.75,  "output":  4.50},
    "gpt-5.4-nano":  {"input": 0.20,  "output":  1.00},

    # --- Chat variants ---
    "gpt-5-chat":    {"input": 1.25,  "output": 10.00},
    "gpt-5.1-chat":  {"input": 1.25,  "output": 10.00},
    "gpt-5.2-chat":  {"input": 1.75,  "output": 14.00},
    "gpt-5.3-chat":  {"input": 1.75,  "output": 14.00},
}

# Fine-tuned model names contain the base model as a prefix (e.g. "ft:gpt-5-mini:...").
# We resolve the pricing by stripping the "ft:" prefix and extracting the base model name.
def resolve_base_model_for_pricing(model_name: str) -> str:
    if model_name.startswith("ft:"):
        parts = model_name.split(":")
        return parts[1] if len(parts) > 1 else model_name
    return model_name


def estimate_request_cost_usd(
    prompt_tokens: int | float | None,
    completion_tokens: int | float | None,
    model_name: str,
) -> float:
    """
    Estimate request cost in USD from token counts and the model name.
    Returns NaN if the model is not in the pricing table.
    """
    base_name = resolve_base_model_for_pricing(model_name)
    pricing = MODEL_PRICING_PER_1M_TOKENS.get(base_name)

    if pricing is None:
        return np.nan

    prompt_tokens     = 0 if pd.isna(prompt_tokens)     else prompt_tokens
    completion_tokens = 0 if pd.isna(completion_tokens) else completion_tokens

    input_cost  = (prompt_tokens     / 1_000_000) * pricing["input"]
    output_cost = (completion_tokens / 1_000_000) * pricing["output"]
    return float(input_cost + output_cost)


def call_openai_chat(client, model_name: str, messages: list[dict], temperature: float = 0.0):
    """
    Send one chat-style request and return a dictionary with the model text and token usage.
    """
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
    )

    output_text = response.choices[0].message.content
    output_text = output_text.strip() if isinstance(output_text, str) else output_text

    usage             = getattr(response, "usage", None)
    prompt_tokens     = getattr(usage, "prompt_tokens",     np.nan) if usage else np.nan
    completion_tokens = getattr(usage, "completion_tokens", np.nan) if usage else np.nan
    total_tokens      = getattr(usage, "total_tokens",      np.nan) if usage else np.nan

    return {
        "llm_output":        output_text,
        "prompt_tokens":     prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens":      total_tokens,
    }


def run_fine_tuned_model_on_dataset(
    dataset: pd.DataFrame,
    system_instruction: str,
    client,
    model_name: str,
    num_repeats: int = 1,
    sleep_seconds: float = 0.2,
):
    """
    Run the fine-tuned model on every row of the dataset, repeating each
    inference num_repeats times. Returns a DataFrame with one row per
    text × repeat.
    """
    rows = []

    for repeat_number in range(1, num_repeats + 1):
        for row_idx, row in tqdm(
            dataset.iterrows(),
            total=len(dataset),
            desc=f"Inference run {repeat_number}/{num_repeats}",
        ):
            messages = [
                {"role": "system", "content": system_instruction},
                {"role": "user",   "content": row["plain"]},
            ]

            try:
                response_info     = call_openai_chat(client, model_name, messages, temperature=0.0)
                llm_output        = response_info["llm_output"]
                prompt_tokens     = response_info["prompt_tokens"]
                completion_tokens = response_info["completion_tokens"]
                total_tokens      = response_info["total_tokens"]
                estimated_cost_usd = estimate_request_cost_usd(
                    prompt_tokens, completion_tokens, model_name
                )
                error_message = None
                status        = "ok"
            except Exception as e:
                llm_output         = None
                prompt_tokens      = np.nan
                completion_tokens  = np.nan
                total_tokens       = np.nan
                estimated_cost_usd = np.nan
                error_message      = str(e)
                status             = "api_error"

            rows.append({
                "model_name":              model_name,
                "repeat_number":           repeat_number,
                "dataset_index":           row_idx,
                "textid":                  row["textid"],
                "plain":                   row["plain"],
                "gold_metaphor_tagged_text": row["metaphor_tagged_text"],
                "llm_output":              llm_output,
                "prompt_tokens":           prompt_tokens,
                "completion_tokens":       completion_tokens,
                "total_tokens":            total_tokens,
                "estimated_cost_usd":      estimated_cost_usd,
                "status":                  status,
                "error_message":           error_message,
            })

            time.sleep(sleep_seconds)

    return pd.DataFrame(rows)


def summarise_experiment_costs(results_table: pd.DataFrame):
    """
    Summarise token usage and estimated cost by model.
    """
    if results_table.empty:
        return pd.DataFrame()

    summary = (
        results_table.groupby("model_name", dropna=False)
        .agg(
            num_requests        =("model_name",          "size"),
            successful_requests =("status",              lambda s: int((s == "ok").sum())),
            prompt_tokens       =("prompt_tokens",       "sum"),
            completion_tokens   =("completion_tokens",   "sum"),
            total_tokens        =("total_tokens",        "sum"),
            estimated_cost_usd  =("estimated_cost_usd",  lambda s: s.sum(min_count=1)),
        )
        .reset_index()
    )

    return summary


## 12. Run the fine-tuned model on the test set

This is the inference loop. For every text in the test set, the notebook will:

1. build a two-message conversation (system instruction + plain text)
2. send it to the fine-tuned model via the OpenAI API
3. save the model output
4. record token usage and an estimated request cost
5. repeat the request as many times as specified in `NUM_REPEATS`

If you set `NUM_REPEATS = 5`, each text will appear in five separate rows in the output file.


In [ ]:
results_df = run_fine_tuned_model_on_dataset(
    dataset=test_df,
    system_instruction=SYSTEM_INSTRUCTION,
    client=client,
    model_name=FINE_TUNED_MODEL_NAME,
    num_repeats=NUM_REPEATS,
    sleep_seconds=SLEEP_BETWEEN_REQUESTS_SECONDS,
)

print(f"Total result rows: {len(results_df)}")
results_df.head()


## 13. Summarise token usage and estimated cost

This table gives a simple cost overview for the inference phase.

It adds up:

- how many API requests were made
- how many input and output tokens were used
- the estimated total cost in US dollars

The estimates come from the pricing table defined earlier in the notebook. 
If you change the model or pricing, update that table first.

**Note:** fine-tuning training costs are separate and are billed by OpenAI directly to your account. They are not reflected here.


In [ ]:
cost_summary_df = summarise_experiment_costs(results_df)
cost_summary_df


## 14. Save the raw model outputs and cost summary to CSV

This section saves the full row-by-row API results, including repeated runs, token usage and estimated cost.


In [ ]:
outputs_csv_path      = OUTPUT_DIR / "llm_outputs_fine_tuned.csv"
cost_summary_csv_path = OUTPUT_DIR / "experiment_cost_summary.csv"

results_df.to_csv(outputs_csv_path, index=False)
cost_summary_df.to_csv(cost_summary_csv_path, index=False)

print(f"Saved model outputs to:  {outputs_csv_path.resolve()}")
print(f"Saved cost summary to:   {cost_summary_csv_path.resolve()}")


## 15. Load the evaluation function from `evaluation.py`

The next cell imports the function we need to evaluate the LLM output (`do_praf()`) against the gold standard manually annotated corpus from the Python script `evaluation.py`.


In [ ]:
spec = importlib.util.spec_from_file_location("evaluation_module", EVALUATION_PATH)
evaluation_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(evaluation_module)

do_praf = evaluation_module.do_praf

print("Loaded do_praf() successfully from:", EVALUATION_PATH.resolve())


## 16. Score each output

The next function applies the evaluator row by row.

It produces a row-level metrics table, where each row corresponds to the fine-tuned model applied to one test-set text in one repeat run.


In [ ]:
def evaluate_predictions(results_table: pd.DataFrame):
    row_metrics = []

    for _, row in results_table.iterrows():
        if pd.isna(row["llm_output"]) or row["llm_output"] is None or str(row["llm_output"]).strip() == "":
            continue

        try:
            metrics = do_praf(
                xml_true=row["gold_metaphor_tagged_text"],
                xml_pred=row["llm_output"],
                tag_name="Metaphor",
            )

            row_metrics.append({
                "model_name":        row["model_name"],
                "repeat_number":     row["repeat_number"],
                "dataset_index":     row["dataset_index"],
                "textid":            row["textid"],
                "status":            row["status"],
                "estimated_cost_usd": row["estimated_cost_usd"],
                "precision":         metrics["precision"],
                "recall":            metrics["recall"],
                "accuracy":          metrics["accuracy"],
                "f1":                metrics["f1"],
                "true_pred_disp":    metrics["true_pred_disp"],
                "true_align_disp":   metrics["true_align_disp"],
            })

        except Exception as e:
            row_metrics.append({
                "model_name":        row["model_name"],
                "repeat_number":     row["repeat_number"],
                "dataset_index":     row["dataset_index"],
                "textid":            row["textid"],
                "status":            row["status"],
                "estimated_cost_usd": row["estimated_cost_usd"],
                "precision":         np.nan,
                "recall":            np.nan,
                "accuracy":          np.nan,
                "f1":                np.nan,
                "true_pred_disp":    np.nan,
                "true_align_disp":   np.nan,
                "evaluation_error":  str(e),
            })

    return pd.DataFrame(row_metrics)


row_level_metrics_df = evaluate_predictions(results_df)
print(f"Scored rows: {len(row_level_metrics_df)}")
row_level_metrics_df.head()


## 17. Find outputs where the model did not produce any `<Metaphor>` tags

In some cases, the evaluation step may produce the following warning:

*UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples.*

### Why does this happen?

This warning occurs when the model does **not predict any metaphors at all** for a given text.

Evaluation metrics like precision are calculated based on the number of predicted positives. If the model predicts zero metaphors, then:

- there are no positive predictions
- the precision formula involves dividing by zero
- the metric becomes undefined

To handle this, the evaluation library sets precision to **0.0** for that case and raises this warning.

This is not a bug — it simply indicates that the model was **too conservative** and failed to identify any metaphors in that example.

### What this section does

This section looks for output rows where the model did **not produce any `<Metaphor>...</Metaphor>` tags** and saves these into a separate output file (`lines_without_metaphor_tags.csv`).

These cases are important to inspect because they often explain:
- drops in **recall** (missed metaphors)
- drops in **precision** (imbalanced predictions across classes)

By identifying these examples, you can better understand whether the fine-tuned model learnt to apply tags or remained overly conservative.


In [ ]:
METAPHOR_TAG_PATTERN = r"<Metaphor>.*?</Metaphor>"

no_metaphor_mask = (
    results_df["status"].eq("ok")
    & results_df["llm_output"].apply(
        lambda x: not bool(re.search(METAPHOR_TAG_PATTERN, str(x), re.DOTALL))
    )
)

missing_metaphor_tags_df = results_df[no_metaphor_mask].copy()

missing_metaphor_summary_df = missing_metaphor_tags_df[[
    "model_name",
    "repeat_number",
    "dataset_index",
    "textid",
    "plain",
    "status",
    "error_message",
]].copy()

missing_metaphor_summary_df.insert(0, "csv_row_index", missing_metaphor_tags_df.index)

missing_metaphor_csv_path = OUTPUT_DIR / "lines_without_metaphor_tags.csv"
missing_metaphor_summary_df.to_csv(missing_metaphor_csv_path, index=False)

print(f"Saved rows without metaphor tags to: {missing_metaphor_csv_path.resolve()}")
print(f"Rows with no <Metaphor>...</Metaphor> tag pair: {len(missing_metaphor_summary_df)}")
missing_metaphor_summary_df.head(20)


## 18. Aggregate the metrics for the fine-tuned model

This summary table gives one average score for the fine-tuned model across all test texts and all repeats.

It also records how many repeats were completed and the total estimated inference cost.


In [ ]:
model_level_metrics_df = (
    row_level_metrics_df.groupby(["model_name"], dropna=False)
    .agg(
        runs_scored          =("model_name",          "size"),
        unique_texts         =("textid",              "nunique"),
        repeats_completed    =("repeat_number",       "nunique"),
        avg_precision        =("precision",           "mean"),
        avg_recall           =("recall",              "mean"),
        avg_accuracy         =("accuracy",            "mean"),
        avg_f1               =("f1",                  "mean"),
        total_estimated_cost_usd=("estimated_cost_usd", lambda s: s.sum(min_count=1)),
    )
    .reset_index()
)

model_level_metrics_df


## 19. Save the evaluation files to CSV

This section saves the results of the evaluation in two files:

- `row_level_metrics.csv`: one row per test text × repeat with individual metric scores
- `model_level_metrics.csv`: one summary row for the fine-tuned model


In [ ]:
row_metrics_csv_path   = OUTPUT_DIR / "row_level_metrics.csv"
model_metrics_csv_path = OUTPUT_DIR / "model_level_metrics.csv"

row_level_metrics_df.to_csv(row_metrics_csv_path, index=False)
model_level_metrics_df.to_csv(model_metrics_csv_path, index=False)

print(f"Saved row-level metrics to:   {row_metrics_csv_path.resolve()}")
print(f"Saved model-level metrics to: {model_metrics_csv_path.resolve()}")


## 20. Inspect a few example predictions

This final table lets you compare:

- the original plain text
- the gold tagged text
- the model output
- the evaluation scores

By default, the cell shows the top 10 rows of the results. Change the number in `.head(10)` to see more.


In [ ]:
example_view_df = (
    results_df.merge(
        row_level_metrics_df,
        on=["model_name", "repeat_number", "dataset_index", "textid"],
        how="left",
        suffixes=("", "_metric"),
    )
)

example_view_df[[
    "model_name",
    "repeat_number",
    "textid",
    "plain",
    "gold_metaphor_tagged_text",
    "llm_output",
    "precision",
    "recall",
    "accuracy",
    "f1",
    "estimated_cost_usd",
]].head(10)


## What the output files mean

After the notebook finishes, you should have these files in the `outputs` folder:

- `fine_tune_train.jsonl` 
  The JSONL training file uploaded to the OpenAI Files API. Each line is one training example.

- `fine_tuned_model_id.txt` 
  The ID of the fine-tuned model (e.g. `ft:gpt-5-mini:org:metaphor:XXXXXXXX`).   Paste this into `FINE_TUNED_MODEL_NAME` if you want to skip re-training and run inference only.

- `llm_outputs_fine_tuned.csv` 
  One row per test text × repeat. Includes the raw model output, token usage, cost estimate   and any API error message.

- `experiment_cost_summary.csv` 
  A compact summary of the number of inference requests, tokens used and estimated total cost by model.

- `lines_without_metaphor_tags.csv` 
  A list of rows where the model output did not contain any `<Metaphor>...</Metaphor>` tag pair.

- `row_level_metrics.csv` 
  One row per successfully evaluated test text × repeat with precision, recall, accuracy and F1.

- `model_level_metrics.csv` 
  One summary row for the fine-tuned model, averaging the evaluation metrics across all   scored rows and summing the estimated inference cost.
